# RAGNav quickstart (offline hybrid retrieval)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/irfanalidv/RAGNav/blob/main/cookbook/ragnav_quickstart.ipynb)

Hybrid **BM25 + sentence-transformers** — no API key. Scans the full SQuAD **validation** split, dedupes by passage text until **50** unique contexts are indexed (early validation is heavy on **Super Bowl 50**). Retrieval uses **on-corpus Super Bowl questions** so the snippets match the index; **gold block id in top-5** uses the same labeling idea as `examples/basic/ragnav_squad_retrieval.py`. **QueryFallback** retries with a rephrased query when the first pass is uncertain.

In [1]:
# Suppress protobuf / MessageFactory noise from some local HF Hub + protobuf stacks (harmless).
import warnings

warnings.filterwarnings("ignore", message=".*MessageFactory.*")

%pip install -q "ragnav[embeddings]" "datasets>=2.14.0"


[notice] A new release of pip is available: 25.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
import warnings

warnings.filterwarnings("ignore", message=".*MessageFactory.*")

from datasets import load_dataset

from ragnav import RAGNavIndex, RAGNavRetriever
from ragnav.ingest.markdown import ingest_markdown_string

# Full validation split: many rows share the same `context` string, so we dedupe until we have
# 50 unique passages. A tiny row window (e.g. validation[:200]) often yields only ~10 unique
# contexts — not a bug in RAGNav, just duplicate-heavy rows.
ds = load_dataset("rajpurkar/squad", split="validation")
seen: set[str] = set()
contexts: list[str] = []
first_gold: list[str] = []
for row in ds:
    ctx = row["context"]
    if ctx in seen:
        continue
    seen.add(ctx)
    answers = (row.get("answers") or {}).get("text") or []
    gold = answers[0] if answers else ""
    contexts.append(ctx)
    first_gold.append(gold)
    if len(contexts) >= 50:
        break

print(f"Loaded {len(contexts)} unique passages (target 50).")


def _gold_block_id(doc_blocks: list, gold: str) -> str:
    paras = [b for b in doc_blocks if b.type == "paragraph"]
    if gold:
        g = gold.lower()
        for b in paras:
            if g in b.text.lower():
                return b.block_id
    return paras[0].block_id if paras else doc_blocks[0].block_id


context_to_gold_block: dict[str, str] = {}
documents = []
blocks: list = []
for i, (ctx, gold) in enumerate(zip(contexts, first_gold)):
    doc, doc_blocks = ingest_markdown_string(ctx, name=f"passage_{i}.md")
    documents.append(doc)
    blocks.extend(doc_blocks)
    context_to_gold_block[ctx] = _gold_block_id(doc_blocks, gold)

index = RAGNavIndex.build(
    documents=documents,
    blocks=blocks,
    use_sentence_transformers=True,
    vector_model="all-MiniLM-L6-v2",
    embed_batch_size=32,
)
if len(contexts) < 50:
    print(f"Warning: only {len(contexts)} unique passages in this split (expected 50).")
print(f"Index ready: {len(index.blocks_by_id)} blocks from {len(documents)} documents")

/Users/irfanalidv/.pyenv/versions/3.12.1/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(
/Users/irfanalidv/.pyenv/versions/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 50 unique passages (target 50).


Index ready: 50 blocks from 50 documents


In [3]:
from typing import Optional

questions = [
    "Which team won Super Bowl 50?",
    "What network broadcast Super Bowl 50?",
    "How much did a Super Bowl 50 ad cost?",
]

_STOP = frozenset(
    "the a an of for in on at to from by is are was were be been being "
    "did does do what which who how much many".split()
)


def _gold_block_for_demo_question(q: str) -> Optional[str]:
    """Pick the indexed passage whose official SQuAD question overlaps the demo wording most."""
    qtok = {w.strip("?").lower() for w in q.split() if w and w.strip("?").lower() not in _STOP}
    best_score, best_ctx = -1, None
    for row in ds:
        ctx = row["context"]
        if ctx not in context_to_gold_block:
            continue
        stok = {
            w.strip("?").lower()
            for w in row["question"].split()
            if w and w.strip("?").lower() not in _STOP
        }
        score = len(qtok & stok)
        if score > best_score:
            best_score = score
            best_ctx = ctx
    return context_to_gold_block.get(best_ctx) if best_ctx else None


retriever = RAGNavRetriever(index=index)

for q in questions:
    gold_bid = _gold_block_for_demo_question(q)
    # top_k=5 gives fusion enough headroom; with 3, BM25+vector tie-breaks can surface a related
    # stadium-hosting passage ahead of the main game summary for "who won" style queries.
    result = retriever.retrieve(
        q,
        top_k=5,
        expand_structure=False,
        expand_graph=False,
    )
    top = result.blocks[0].text[:160].replace("\n", " ") if result.blocks else "(no result)"
    in_top5 = gold_bid is not None and gold_bid in {b.block_id for b in result.blocks[:5]}
    print(f"Q: {q}")
    print(f"Top snippet: {top}...")
    print(f"Confidence: {result.confidence.value}")
    print(f"Gold block id in top-5: {in_top5}")
    print()

Q: Which team won Super Bowl 50?
Top snippet: Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conferenc...
Confidence: medium
Gold block id in top-5: True



Q: What network broadcast Super Bowl 50?
Top snippet: In the United States, the game was televised by CBS, as part of a cycle between the three main broadcast television partners of the NFL. The network's lead broa...
Confidence: medium
Gold block id in top-5: True



Q: How much did a Super Bowl 50 ad cost?
Top snippet: CBS set the base rate for a 30-second advertisement at $5,000,000, a record high price for a Super Bowl ad. As of January 26, the advertisements had not yet sol...
Confidence: medium
Gold block id in top-5: True



## QueryFallback — automatic retry on low / medium confidence

`FakeLLMClient` echoes prompts, so it is a poor query rewriter. For this demo we use a tiny stub that returns **one** alternative line when the fallback asks for variations. The vague query and the rephrase are both about **Super Bowl 50**, which appears in this notebook’s SQuAD slice, so the retrieved text stays on-corpus. In production, plug in any `LLMClient` (e.g. Mistral with `pip install ragnav[mistral]`).

In [4]:
from ragnav.retrieval import QueryFallback


class DemoVariationLLM:
    """Colab-only: return a single rephrasing line for the variation prompt."""

    def chat(self, *, messages, model=None, temperature=0.7):
        # Align with SQuAD content in this notebook (commercials / Super Bowl 50).
        return "What was the cost of a 30-second Super Bowl 50 ad?\n"


fallback = QueryFallback(retriever=retriever, llm=DemoVariationLLM())

# Vague wording first; rephrase targets an on-corpus question (see passage_specs / SQuAD).
out = fallback.retrieve(
    "SB50 TV commercial price",
    top_k=5,
    expand_structure=False,
    expand_graph=False,
)
print("queries_tried:", out.queries_tried)
print("improved:", out.improved)
print("final confidence:", out.final_result.confidence.value)
if out.final_result.blocks:
    print("top block:", out.final_result.blocks[0].text[:200].replace("\n", " "))

queries_tried: ['SB50 TV commercial price', 'What was the cost of a 30-second Super Bowl 50 ad?', 'What was the cost of a 30-second Super Bowl 50 ad?']
improved: False
final confidence: medium
top block: CBS broadcast Super Bowl 50 in the U.S., and charged an average of $5 million for a 30-second commercial during the game. The Super Bowl 50 halftime show was headlined by the British rock group Coldpl
